<a href="https://colab.research.google.com/github/paulaprado1904/AlgoritmoEvolutivo/blob/main/AEMMT_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import math
import time
import numpy as np
import pandas as pd
import os
from datetime import datetime
import matplotlib.pyplot as plt

# Contador global de avaliações da função objetivo.
# É usado para controlar o esforço computacional total do algoritmo.
GLOBAL_AVALIACOES = 0


# ======================================================
# === Estrutura do indivíduo (representação absoluta) ===
# ======================================================
class Individuo:
    """
    Representa uma solução candidata para o problema de dobramento proteico
    no modelo HP em 3D.

    Cada indivíduo é codificado por uma sequência de movimentos absolutos
    em uma malha cúbica 3D. A partir dessa sequência, são construídas as
    coordenadas espaciais da proteína e calculadas as métricas usadas pelo
    algoritmo evolutivo.
    """
    def __init__(self, movimentos):
        # Lista de movimentos absolutos no espaço 3D
        self.movimentos = movimentos[:]

        # Coordenadas resultantes da construção da cadeia
        self.coords = None

        # Número de colisões (sobreposição de resíduos)
        self.nC = 0

        # Número de contatos hidrofóbicos H-H não consecutivos
        self.nH = 0

        # Função objetivo principal
        self.f = None

        # Métricas de compactação
        self.dmax = None
        self.davg = None

        # Métrica combinada usada em uma das tabelas/subpopulações
        self.delta = None

        # Energia simplificada baseada em distâncias entre resíduos H
        self.energiaSimplif = None

        # Energia final associada à solução
        self.energia = None

    def __str__(self):
        """
        Retorna uma representação textual resumida do indivíduo,
        útil para depuração e inspeção dos resultados.
        """
        f_val = self.f if self.f is not None else 0
        d_max = self.dmax if self.dmax is not None else 0
        d_avg = self.davg if self.davg is not None else 0
        delta_val = self.delta if self.delta is not None else 0
        energia_val = self.energia if self.energia is not None else 0
        energia_simplif_val = (
            self.energiaSimplif if self.energiaSimplif is not None else 0
        )

        return (
            f"f(c): {f_val:3} | "
            f"nC: {self.nC:2} | "
            f"dmax: {d_max:5.2f} | "
            f"davg: {d_avg:5.2f} | "
            f"delta: {delta_val:6.2f} | "
            f"Energia: {energia_val} | "
            f"energiaSimplif: {energia_simplif_val:6.2f} | "
            f"Movimentos: {self.movimentos}"
        )


# ======================================================
# === Movimentos absolutos em 3D (U, L, F, B, R, D) ===
# ======================================================
def delta_from_move_3d(mov):
    """
    Converte o código inteiro de um movimento em seu deslocamento
    cartesiano correspondente na malha 3D.

    Convenção adotada:
    0 -> U : +x
    1 -> L : +y
    2 -> F : +z
    3 -> B : -z
    4 -> R : -y
    5 -> D : -x
    """
    if mov == 0:
        return (1, 0, 0)
    if mov == 1:
        return (0, 1, 0)
    if mov == 2:
        return (0, 0, 1)
    if mov == 3:
        return (0, 0, -1)
    if mov == 4:
        return (0, -1, 0)
    if mov == 5:
        return (-1, 0, 0)
    raise ValueError("mov inválido (esperado 0..5)")


# ======================================================
# === Construção das coordenadas e contagem de colisões ===
# ======================================================
def construir_coordenadas(ind):
    """
    Constrói a trajetória espacial do indivíduo a partir da sua sequência
    de movimentos absolutos.

    Retorna:
    - coords: lista de coordenadas ocupadas pela cadeia
    - nC: número de posições com sobreposição (colisões)

    Observação:
    uma colisão ocorre quando dois ou mais resíduos ocupam o mesmo ponto
    da malha, tornando a conformação inviável.
    """
    x = y = z = 0
    coords = [(0, 0, 0)]
    ocupacao = {(0, 0, 0): 1}

    for mov in ind.movimentos:
        dx, dy, dz = delta_from_move_3d(mov)
        x += dx
        y += dy
        z += dz
        pt = (x, y, z)
        coords.append(pt)
        ocupacao[pt] = ocupacao.get(pt, 0) + 1

    nC = sum(1 for v in ocupacao.values() if v > 1)

    ind.coords = coords
    ind.nC = nC
    return coords, nC


def avaliar_base(ind, cadeia, rho=1.0):
    """
    Avalia completamente um indivíduo.

    Métricas calculadas:
    - nC: número de colisões
    - nH: número de contatos H-H não adjacentes na cadeia
    - f(c): função objetivo principal
    - dmax: maior distância entre pares H-H não consecutivos
    - davg: média das distâncias entre pares H-H não consecutivos
    - delta: métrica combinada calculada posteriormente
    - energiaSimplif: soma das distâncias entre resíduos H
    - energia: energia final usada no acompanhamento da execução

    Regras principais:
    - Se houver colisão, o indivíduo é considerado inviável.
    - Soluções inviáveis recebem penalização proporcional ao número de colisões.
    - Se não houver colisão, calcula-se o número de contatos H-H.
    """
    global GLOBAL_AVALIACOES
    GLOBAL_AVALIACOES += 1

    PENAL = 1e6

    coords, nC = construir_coordenadas(ind)
    n = len(cadeia)

    # Salva as coordenadas no indivíduo para reutilização posterior
    ind.coords = coords

    # Caso inconsistente: conformação inválida
    if coords is None or len(coords) != n:
        ind.nC = max(1, int(nC) if nC is not None else 1)
        ind.nH = 0
        ind.f = -rho * ind.nC
        ind.dmax = float('inf')
        ind.davg = float('inf')
        ind.delta = None
        ind.energia = -ind.f
        return

    # --------------------------------------------------
    # Cálculo das distâncias entre pares de resíduos H
    # não consecutivos na cadeia
    # --------------------------------------------------
    H_idx = [i for i, a in enumerate(cadeia) if a == "H"]
    dists = []
    m = len(H_idx)
    Es = 0.0

    for a in range(m):
        i = H_idx[a]
        for b in range(a + 1, m):
            j = H_idx[b]

            # Ignora pares consecutivos, pois já estão ligados covalentemente
            if abs(i - j) == 1:
                continue

            p1 = coords[i]
            p2 = coords[j]

            # Se ocupam exatamente a mesma posição, ignora neste cálculo
            if p1 == p2:
                continue

            d = math.sqrt(
                (p1[0] - p2[0]) ** 2 +
                (p1[1] - p2[1]) ** 2 +
                (p1[2] - p2[2]) ** 2
            )

            dists.append(d)
            Es += d

    if not dists:
        dmax = float('inf')
        davg = float('inf')
    else:
        dmax = max(dists)
        davg = sum(dists) / len(dists)

    # --------------------------------------------------
    # Caso inviável: há colisões
    # --------------------------------------------------
    if nC > 0:
        ind.nC = nC
        ind.nH = 0
        ind.f = -rho * nC

        # Mesmo em soluções inviáveis, ainda se mantém informação
        # relacionada à compactação, incorporando penalização
        ind.dmax = 1 / (dmax + rho * nC)
        ind.davg = 1 / (davg + rho * nC)

        ind.delta = None
        Es = Es + (PENAL * nC)
        ind.energiaSimplif = Es
        ind.energia = -ind.f
        return

    # --------------------------------------------------
    # Caso viável: sem colisões
    # Conta contatos H-H não consecutivos e adjacentes no espaço
    # --------------------------------------------------
    nH = 0
    for i in range(n):
        if cadeia[i] != "H":
            continue
        for j in range(i + 2, n):
            if cadeia[j] != "H":
                continue

            pi = coords[i]
            pj = coords[j]
            manh = abs(pi[0] - pj[0]) + abs(pi[1] - pj[1]) + abs(pi[2] - pj[2])

            if manh == 1:
                nH += 1

    ind.nC = 0
    ind.nH = nH
    ind.f = nH
    ind.dmax = 1 / dmax
    ind.davg = 1 / davg
    ind.delta = None
    ind.energiaSimplif = Es
    ind.energia = -ind.f


# ======================================================
# === Atualização da métrica δ(c) ===
# ======================================================
def atualizar_delta(populacao, prev_stats):
    """
    Atualiza a métrica delta de cada indivíduo da população.

    A ideia é escolher dinamicamente entre dmax e davg com base
    na variação média observada entre gerações consecutivas.

    Depois de escolher a métrica ativa, define:
        delta(c) = f(c) * d(c)

    em que d(c) pode ser dmax ou davg, dependendo da geração.
    """
    dmax_vals = np.array([ind.dmax for ind in populacao], dtype=float)
    davg_vals = np.array([ind.davg for ind in populacao], dtype=float)

    dmax_fin = dmax_vals[np.isfinite(dmax_vals)]
    davg_fin = davg_vals[np.isfinite(davg_vals)]

    mean_dmax = float(np.mean(dmax_fin)) if dmax_fin.size else float("inf")
    mean_davg = float(np.mean(davg_fin)) if davg_fin.size else float("inf")

    prev_dmax = float(prev_stats.get("mean_dmax", float("inf"))) if prev_stats else float("inf")
    prev_davg = float(prev_stats.get("mean_davg", float("inf"))) if prev_stats else float("inf")
    active_metric_prev = prev_stats.get("active_metric", "dmax") if prev_stats else "dmax"

    def razao(atual, anterior):
        """
        Mede o quanto a métrica mudou em relação à geração anterior.
        Retorna |atual/anterior - 1|.
        """
        if (not np.isfinite(atual)) or (not np.isfinite(anterior)) or anterior <= 0:
            return None
        return abs(atual / anterior - 1)

    r_dmax = razao(mean_dmax, prev_dmax)
    r_davg = razao(mean_davg, prev_davg)

    active_metric = escolhe_metrica_por_variacao(r_dmax, r_davg, atual=active_metric_prev)

    for ind in populacao:
        if ind.f is None or not np.isfinite(ind.f):
            ind.delta = None
            continue

        d_ind = ind.dmax if active_metric == "dmax" else ind.davg
        if (d_ind is None) or (not np.isfinite(d_ind)) or d_ind <= 0:
            ind.delta = None
            continue

        ind.delta = ind.f * d_ind

        # Informação auxiliar para depuração
        ind.d = d_ind

    return {
        "mean_dmax": mean_dmax,
        "mean_davg": mean_davg,
        "active_metric": active_metric,
        "r_dmax": r_dmax,
        "r_davg": r_davg
    }


def escolhe_metrica_por_variacao(r_dmax, r_davg, atual="dmax"):
    """
    Escolhe qual métrica de compactação será usada na geração atual.

    Critério:
    - escolhe a métrica que apresentou maior variação relativa;
    - em caso de empate, mantém a métrica usada anteriormente.
    """
    if r_dmax is None and r_davg is None:
        return atual
    if r_dmax is None:
        return "davg"
    if r_davg is None:
        return "dmax"

    var_dmax = abs(r_dmax - 1.0)
    var_davg = abs(r_davg - 1.0)

    if abs(var_dmax - var_davg) < 1e-15:
        return atual

    return "dmax" if var_dmax > var_davg else "davg"


# ======================================================
# === Geração aleatória de indivíduo ===
# ======================================================
def gerar_individuo_aleatorio(n_residuos):
    """
    Gera um indivíduo aleatório com n_residuos - 1 movimentos.

    Observação:
    o movimento 3 (-z) foi excluído da geração inicial para reduzir
    simetrias redundantes na construção das conformações.
    """
    L = n_residuos - 1
    moves = [0, 1, 2, 4, 5]
    movimentos = [random.choice(moves) for _ in range(L)]
    return Individuo(movimentos)


# ======================================================
# === Recombinação por múltiplos pontos ===
# ======================================================
def crossover_multipontos_2filhos(pai, mae, n_residuos):
    """
    Realiza crossover de múltiplos pontos entre dois pais
    e retorna dois filhos.

    O número de cortes é definido proporcionalmente ao tamanho
    da proteína.
    """
    L = len(pai.movimentos)
    if L != len(mae.movimentos):
        raise ValueError("Pais com comprimentos diferentes de cromossomo")

    num_cortes = int(round(n_residuos / 10.0))
    num_cortes = max(1, min(num_cortes, L - 1))
    cortes = sorted(random.sample(range(1, L), num_cortes))

    def build_child(start_with_pai: bool):
        filho_mov = []
        pais = [pai.movimentos, mae.movimentos]
        idx = 0 if start_with_pai else 1
        ini = 0

        for c in cortes + [L]:
            filho_mov.extend(pais[idx][ini:c])
            idx = 1 - idx
            ini = c

        return Individuo(filho_mov)

    return build_child(True), build_child(False)


# ======================================================
# === Mutação gene a gene ===
# ======================================================
def mutacao_por_gene(ind, taxa_mut):
    """
    Aplica mutação gene a gene.

    Cada posição do cromossomo é alterada com probabilidade taxa_mut,
    substituindo o movimento atual por um novo valor aleatório.
    """
    novo = Individuo(ind.movimentos[:])

    for j in range(len(novo.movimentos)):
        if random.random() <= taxa_mut:
            novo.movimentos[j] = random.randint(0, 5)

    return novo


# ======================================================
# === Seleção de pais ===
# ======================================================
def selecao_pai_pool_unico(tabelas, k=3):
    """
    Une todos os indivíduos presentes nas tabelas, remove duplicatas
    por referência de objeto e realiza seleção por torneio usando delta.
    """
    pool = unir_tabelas(*[t for t in tabelas if t])
    if not pool:
        raise RuntimeError("Pool vazio: todas as tabelas estão vazias.")

    pool_validos = [ind for ind in pool if getattr(ind, "delta", None) is not None]

    if len(pool_validos) >= k:
        return selecao_torneio(pool_validos, k=k, atributo="delta", maximize=True)

    if pool_validos:
        return selecao_torneio(pool_validos, k=len(pool_validos), atributo="delta", maximize=True)

    return random.choice(pool)


def selecao_pai_pool_unico2(tabelas, k=3):
    """
    Variante simplificada de seleção:
    une todas as tabelas e escolhe um indivíduo aleatoriamente.
    """
    pool = unir_tabelas(*[t for t in tabelas if t])
    if not pool:
        raise RuntimeError("Pool vazio: todas as tabelas estão vazias.")
    return random.choice(pool)


def selecao_torneio(pop, k=3, atributo="delta", maximize=True):
    """
    Realiza seleção por torneio com base em um atributo do indivíduo.
    """
    if not pop:
        raise RuntimeError("População vazia no torneio.")

    cand = random.sample(pop, k=min(k, len(pop)))
    cand = [ind for ind in cand if getattr(ind, atributo, None) is not None]

    if not cand:
        return random.choice(pop)

    key_fn = lambda ind: getattr(ind, atributo)
    return max(cand, key=key_fn) if maximize else min(cand, key=key_fn)


def assinatura_ind(ind: Individuo):
    """
    Gera uma assinatura imutável do indivíduo com base no cromossomo.
    É usada para detectar genótipos repetidos.
    """
    return tuple(ind.movimentos)


# ======================================================
# === Inserção nas tabelas/subpopulações ===
# ======================================================
def inserir_tabela(tabela, ind, tamanho, maximize, atributo, limite_piora_f=0):
    """
    Insere um indivíduo em uma tabela especializada, respeitando:
    - unicidade por cromossomo;
    - tamanho máximo da tabela;
    - critério de maximização ou minimização do atributo.
    """
    sig = assinatura_ind(ind)
    if any(assinatura_ind(x) == sig for x in tabela):
        return

    val = getattr(ind, atributo, None)
    if val is None:
        return

    if len(tabela) < tamanho:
        tabela.append(ind)
        return

    if maximize:
        pior = min(tabela, key=lambda x: getattr(x, atributo))
        melhor_no_attr = val > getattr(pior, atributo)
    else:
        pior = max(tabela, key=lambda x: getattr(x, atributo))
        melhor_no_attr = val < getattr(pior, atributo)

    if not melhor_no_attr:
        return

    tabela.remove(pior)
    tabela.append(ind)


def unir_tabelas(*tabelas):
    """
    Une várias tabelas em uma única lista sem repetir objetos.
    """
    uniao = []
    seen = set()

    for tab in tabelas:
        for ind in tab:
            if id(ind) not in seen:
                seen.add(id(ind))
                uniao.append(ind)

    return uniao


def pool_unico_por_tabelas(tabelas):
    """
    Cria um pool único sem duplicação por cromossomo.
    """
    pool = []
    seen = set()

    for tab in tabelas:
        for ind in tab:
            key = tuple(ind.movimentos)
            if key not in seen:
                seen.add(key)
                pool.append(ind)

    return pool


def avaliar_parcial_nH_nC(ind, cadeia):
    """
    Avaliação parcial usada antes da avaliação completa.

    Calcula apenas:
    - nC: número de colisões
    - nH: contatos H-H, se a solução não tiver colisões

    Essa função é usada como filtro barato para decidir se vale a pena
    mutar ou aceitar um descendente, sem incrementar o contador global
    de avaliações completas.
    """
    coords, nC = construir_coordenadas(ind)
    n = len(cadeia)

    if coords is None or len(coords) != n:
        return 1, 0

    if nC > 0:
        return nC, 0

    nH = 0
    for i in range(n):
        if cadeia[i] != "H":
            continue
        for j in range(i + 2, n):
            if cadeia[j] != "H":
                continue

            pi = coords[i]
            pj = coords[j]
            manh = abs(pi[0] - pj[0]) + abs(pi[1] - pj[1]) + abs(pi[2] - pj[2])

            if manh == 1:
                nH += 1

    return 0, nH


def estatisticas_pool(pool):
    """
    Calcula estatísticas do pool final:
    - número de genótipos únicos
    - energia média
    - desvio padrão da energia
    """
    if not pool:
        return 0, float("nan"), float("nan")

    n_unicos = len({tuple(ind.movimentos) for ind in pool})

    E = np.array([ind.energia for ind in pool], dtype=float)
    E = E[np.isfinite(E)]

    if E.size == 0:
        return n_unicos, float("nan"), float("nan")

    E_media = float(np.mean(E))
    E_dp = float(np.std(E))
    return n_unicos, E_media, E_dp


def salvar_grafico_melhor_energia_com_marcador(df_hist, pasta_saida, nome_proteina, execucao_id):
    """
    Salva um gráfico da evolução da melhor energia ao longo das gerações,
    destacando a geração em que ocorreu a melhor solução da execução.
    """
    if df_hist is None or df_hist.empty:
        print("[WARN] df_hist vazio.")
        return None

    if "Iteracao" not in df_hist.columns or "Melhor Energia" not in df_hist.columns:
        print("[WARN] df_hist sem colunas necessárias (Iteracao, Melhor Energia).")
        return None

    xs = df_hist["Iteracao"].to_numpy()
    ys = df_hist["Melhor Energia"].to_numpy()

    idx_best = int(df_hist["Melhor Energia"].idxmin())
    best_gen = int(df_hist.loc[idx_best, "Iteracao"])
    best_E = float(df_hist.loc[idx_best, "Melhor Energia"])

    plt.figure(figsize=(8, 4.5))
    plt.plot(xs, ys)
    plt.axvline(best_gen, linestyle="--", linewidth=1)
    plt.scatter([best_gen], [best_E], s=60)

    plt.xlabel("Geração")
    plt.ylabel("Melhor energia")
    plt.title(f"{nome_proteina} — exec {execucao_id} — melhor gen {best_gen} (E={best_E:.0f})")
    plt.grid(True, alpha=0.3)

    ultima_gen = int(df_hist["Iteracao"].max())
    nome_arquivo = f"{nome_proteina}_exec{execucao_id:02d}_ate{ultima_gen:04d}_bestgen{best_gen:04d}.png"
    caminho_fig = os.path.join(pasta_saida, nome_arquivo)

    plt.tight_layout()
    plt.savefig(caminho_fig, dpi=200)
    plt.close()
    return caminho_fig


# ======================================================
# === Algoritmo AEMT 3D sem Monte Carlo ===
# ======================================================
def aemt_hp_3d_sem_mc(
    cadeia_raw,
    pop_init=10,
    taxa_mut=0.2,
    rho=1.0,
    subpop_sizes=(1, 2, 2, 3, 2),
    torneio_k=2,
    taxa_cross=1.0,
    energia_otima=None,
    max_avaliacoes=None,
    seed=None
):
    """
    Executa o algoritmo evolutivo multiobjetivo baseado em tabelas (AEMT)
    para o problema de dobramento proteico HP em 3D.

    Estrutura geral:
    1. Gera população inicial
    2. Avalia indivíduos
    3. Distribui indivíduos em subpopulações especializadas
    4. Gera descendentes por seleção, crossover e mutação
    5. Atualiza as tabelas com os novos indivíduos
    6. Repete até atingir o limite de avaliações ou o ótimo conhecido

    Retorna:
    - df_hist: histórico por geração
    - resumo: principais estatísticas da execução
    - tabelas_finais: tabelas especializadas ao final
    - pool_final: união única das tabelas finais
    """
    global GLOBAL_AVALIACOES

    cadeia = ''.join([c for c in cadeia_raw.strip().upper() if c in ('H', 'P')])
    n_res = len(cadeia)

    if n_res < 2:
        raise ValueError("Cadeia muito curta")

    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    # ---------------- População inicial ----------------
    populacao = []
    for _ in range(pop_init):
        if max_avaliacoes is not None and GLOBAL_AVALIACOES >= max_avaliacoes:
            break

        ind = gerar_individuo_aleatorio(n_res)
        avaliar_base(ind, cadeia, rho=rho)
        populacao.append(ind)

    if len(populacao) == 0:
        raise RuntimeError("Nenhum indivíduo pôde ser avaliado (max_avaliacoes muito baixo).")

    prev_stats = atualizar_delta(populacao, prev_stats=None)

    size_f, size_dmax, size_davg, size_delta, size_simplif = subpop_sizes
    T_f, T_dmax, T_davg, T_delta, T_simplif = [], [], [], [], []

    for ind in populacao:
        inserir_tabela(T_f, ind, size_f, True, 'f')
        inserir_tabela(T_dmax, ind, size_dmax, True, 'dmax')
        inserir_tabela(T_davg, ind, size_davg, True, 'davg')
        inserir_tabela(T_delta, ind, size_delta, True, 'delta')
        inserir_tabela(T_simplif, ind, size_simplif, False, 'energiaSimplif')

    melhores_energia = []
    medias_energia = []
    desvios_energia = []

    melhor_global = min(populacao, key=lambda ind: ind.energia)
    geracao_melhor = 0
    hit_otimo = False
    aval_ate_otimo = None

    if energia_otima is not None and melhor_global.energia <= energia_otima:
        hit_otimo = True
        aval_ate_otimo = GLOBAL_AVALIACOES

    geracao = 0
    while True:
        if max_avaliacoes is not None and GLOBAL_AVALIACOES >= max_avaliacoes:
            break
        if energia_otima is not None and hit_otimo:
            break

        tabelas = [T_f, T_dmax, T_davg, T_delta, T_simplif]

        nova_pop = []
        assinaturas = set()

        while len(nova_pop) < pop_init:
            if max_avaliacoes is not None and GLOBAL_AVALIACOES >= max_avaliacoes:
                break
            if energia_otima is not None and hit_otimo:
                break

            pai = selecao_pai_pool_unico(tabelas, 3)
            mae = selecao_pai_pool_unico2(tabelas, 3)

            if random.random() < taxa_cross:
                base_f1, base_f2 = crossover_multipontos_2filhos(pai, mae, n_res)
            else:
                base_f1, base_f2 = Individuo(pai.movimentos[:]), Individuo(mae.movimentos[:])

            for base in (base_f1, base_f2):
                if len(nova_pop) >= pop_init:
                    break

                # Filho base antes da possível mutação
                cand0 = Individuo(base.movimentos[:])

                # Avaliação parcial barata para decidir se vale mutar
                nC0, nH0 = avaliar_parcial_nH_nC(cand0, cadeia)

                best_nH = melhor_global.nH if getattr(melhor_global, "nH", None) is not None else 0

                # Só evita mutação se o filho parcial já for promissor
                precisa_mutar = not (nH0 > best_nH)

                filho_final = None

                if not precisa_mutar:
                    sig0 = assinatura_ind(cand0)
                    if sig0 not in assinaturas:
                        filho_final = cand0
                        assinaturas.add(sig0)
                    else:
                        precisa_mutar = True

                if precisa_mutar:
                    for _tent in range(10):
                        cand = mutacao_por_gene(cand0, taxa_mut)
                        sig = assinatura_ind(cand)
                        if sig not in assinaturas:
                            filho_final = cand
                            assinaturas.add(sig)
                            break

                    if filho_final is None:
                        filho_final = mutacao_por_gene(cand0, taxa_mut)
                        sig = assinatura_ind(filho_final)
                        assinaturas.add(sig)

                # Avaliação completa: conta no orçamento global de avaliações
                avaliar_base(filho_final, cadeia, rho=rho)
                nova_pop.append(filho_final)

                if energia_otima is not None and not hit_otimo:
                    if filho_final.energia <= energia_otima:
                        hit_otimo = True
                        aval_ate_otimo = GLOBAL_AVALIACOES
                        melhor_global = filho_final
                        geracao_melhor = geracao + 1

        if len(nova_pop) == 0:
            break

        prev_stats = atualizar_delta(nova_pop, prev_stats)

        for ind in nova_pop:
            inserir_tabela(T_f, ind, size_f, True, 'f')
            inserir_tabela(T_dmax, ind, size_dmax, True, 'dmax')
            inserir_tabela(T_davg, ind, size_davg, True, 'davg')
            inserir_tabela(T_delta, ind, size_delta, True, 'delta')
            inserir_tabela(T_simplif, ind, size_simplif, False, 'energiaSimplif')

        energias = [ind.energia for ind in nova_pop]
        medias_energia.append(float(np.mean(energias)))
        desvios_energia.append(float(np.std(energias)))

        melhor_ger = min(nova_pop, key=lambda ind: ind.energia)
        melhores_energia.append(melhor_ger.energia)

        if melhor_ger.energia < melhor_global.energia:
            melhor_global = melhor_ger
            geracao_melhor = geracao + 1

            if energia_otima is not None and melhor_global.energia <= energia_otima and not hit_otimo:
                hit_otimo = True
                aval_ate_otimo = GLOBAL_AVALIACOES

        populacao = nova_pop
        geracao += 1
        print(f"Geracao: {geracao}")

    resumo = {
        "Menor Energia Obtida": melhor_global.energia,
        "Melhor f(c)": melhor_global.f,
        "nH(c)": melhor_global.nH,
        "nC(c)": melhor_global.nC,
        "dmax(c)": melhor_global.dmax,
        "davg(c)": melhor_global.davg,
        "delta(c)": melhor_global.delta,
        "Geração do Melhor": geracao_melhor,
        "Hit_otimo": hit_otimo,
        "Aval_ate_otimo": aval_ate_otimo,
        "Aval_total": GLOBAL_AVALIACOES,
        "Métrica δ ativa última geração": prev_stats["active_metric"]
    }

    df_hist = pd.DataFrame({
        "Iteracao": list(range(len(melhores_energia))),
        "Energia Média": medias_energia,
        "Desvio Padrão": desvios_energia,
        "Melhor Energia": melhores_energia
    })

    tabelas_finais = (T_f, T_dmax, T_davg, T_delta, T_simplif)
    pool_final = pool_unico_por_tabelas(tabelas_finais)

    return df_hist, resumo, tabelas_finais, pool_final


# ======================================================
# === Bloco principal de execução experimental ===
# ======================================================
if __name__ == "__main__":
    caminho_arquivo = "/content/drive/MyDrive/pastaDestino/273d.6.txt"
    pasta_saida = "/content/drive/MyDrive/pastaDestino/273d.6.txt"
    os.makedirs(pasta_saida, exist_ok=True)

    # Leitura do arquivo com:
    # - primeira linha: tamanho da cadeia
    # - segunda linha: sequência HP
    with open(caminho_arquivo) as f:
        n = int(f.readline().strip())
        cadeia = ''.join([c for c in f.readline().strip().upper() if c in ('H', 'P')])
        assert len(cadeia) == n

    nome_proteina = os.path.splitext(os.path.basename(caminho_arquivo))[0]

    # Ótimo conhecido para a instância analisada
    ENERGIA_OTIMA = -12

    # Ajuste de parâmetros conforme o tamanho da proteína
    if n <= 36:
        subpop_sizes = (100, 50, 50, 250, 50)
        taxa_mut = 0.1
        taxa_cross = 0.95
        MAX_AVALIACOES = 1_500_000
    else:
        # Atenção:
        # aqui o ideal é manter 5 valores, pois o algoritmo trabalha com 5 tabelas.
        subpop_sizes = (1, 1, 1, 2, 1)
        taxa_mut = 0.40
        taxa_cross = 0.50
        MAX_AVALIACOES = 3_000_000

    NUM_EXECUCOES = 1
    resultados_execucoes = []

    for i in range(NUM_EXECUCOES):
        GLOBAL_AVALIACOES = 0
        random.seed()
        np.random.seed()

        print(f"\n========== EXECUÇÃO {i+1}/{NUM_EXECUCOES} ==========")

        df_hist, resumo, tabelas, pool_final = aemt_hp_3d_sem_mc(
            cadeia,
            pop_init=500,
            taxa_mut=taxa_mut,
            rho=1.0,
            subpop_sizes=subpop_sizes,
            torneio_k=1,
            taxa_cross=taxa_cross,
            energia_otima=ENERGIA_OTIMA,
            max_avaliacoes=MAX_AVALIACOES
        )

        n_unicos_pool, E_media_pool, E_dp_pool = estatisticas_pool(pool_final)

        caminho_fig = salvar_grafico_melhor_energia_com_marcador(
            df_hist, pasta_saida, nome_proteina, i + 1
        )
        print(f"[OK] Execução {i+1}: gráfico salvo em {caminho_fig}")

        resultados_execucoes.append({
            "Execucao": i + 1,
            "Melhor_Energia": resumo["Menor Energia Obtida"],
            "Hit_otimo": resumo["Hit_otimo"],
            "Aval_ate_otimo": resumo["Aval_ate_otimo"],
            "Aval_total": resumo["Aval_total"],
            "Genotipos_unicos_final_pool": n_unicos_pool,
            "Energia_media_final_pool": E_media_pool,
            "Energia_dp_final_pool": E_dp_pool,
        })

    df_resumo = pd.DataFrame(resultados_execucoes)

    # Estatísticas agregadas da melhor energia por execução
    energies = df_resumo["Melhor_Energia"].replace([np.inf, -np.inf], np.nan).dropna()

    melhor_energia_media = float(energies.mean()) if not energies.empty else float("nan")
    melhor_energia_dp = float(energies.std(ddof=1)) if len(energies) > 1 else float("nan")

    qtd_facteis = int(energies.shape[0])
    qtd_total = int(df_resumo.shape[0])

    print("\n\n===== RESUMO DAS EXECUÇÕES =====")
    print(df_resumo)

    df_sucesso = df_resumo[df_resumo["Hit_otimo"] == True]

    if not df_sucesso.empty:
        media_aval = df_sucesso["Aval_ate_otimo"].mean()
        mediana_aval = df_sucesso["Aval_ate_otimo"].median()
    else:
        media_aval = float('nan')
        mediana_aval = float('nan')

    taxa_acertos = 100.0 * df_sucesso.shape[0] / NUM_EXECUCOES

    print("\n===== RESUMO GLOBAL =====")
    print(f"Ótimo conhecido: {ENERGIA_OTIMA}")
    print(f"Menor energia obtida: {df_resumo['Melhor_Energia'].min()}")
    print(f"Acurácia (Ac.): {taxa_acertos:.1f} %")
    print(f"Média Aval. (apenas execuções com ótimo): {media_aval:.0f}")
    print(f"Mediana Aval.: {mediana_aval:.0f}")

    # Salva o resumo em arquivo TXT
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    caminho_saida_txt = os.path.join(
        pasta_saida,
        f"{nome_proteina}_{timestamp}.txt"
    )

    with open(caminho_saida_txt, "w") as f_out:
        f_out.write(f"Arquivo da proteína: {caminho_arquivo}\n")
        f_out.write(f"Nome da proteína (base): {nome_proteina}\n")
        f_out.write(f"Ótimo conhecido: {ENERGIA_OTIMA}\n\n")

        f_out.write("===== RESUMO DAS EXECUÇÕES =====\n")
        f_out.write(df_resumo.to_string(index=False))
        f_out.write("\n\n===== RESUMO GLOBAL =====\n")
        f_out.write(f"Menor energia obtida: {df_resumo['Melhor_Energia'].min()}\n")
        f_out.write(f"Acurácia (Ac.): {taxa_acertos:.1f} %\n")
        f_out.write(f"Média Aval. (execuções com ótimo): {media_aval:.0f}\n")
        f_out.write(f"Mediana Aval.: {mediana_aval:.0f}\n")

        f_out.write("\n----- ESTATÍSTICAS (MELHOR ENERGIA POR EXECUÇÃO) -----\n")
        f_out.write(f"Melhor_Energia (média ± dp): {melhor_energia_media:.4f} ± {melhor_energia_dp:.4f}\n")
        f_out.write(f"Execuções factíveis consideradas: {qtd_facteis}/{qtd_total}\n")

    print(f"\nResumo salvo em: {caminho_saida_txt}")